# Nomes e sobrenomes do Brasil: exploratory analysis

A tour of the two tables: real frequencies (CNPJ register and IBGE 2010 Census), the six matching keys, and the meanings written by OpenAI models in 2023, which are often wrong. Source and documentation: [github.com/CaiqueDourado/nomes-e-sobrenomes-brasil](https://github.com/CaiqueDourado/nomes-e-sobrenomes-brasil).

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 160)
# Works in the repository (data/) and on Kaggle (/kaggle/input/...)
candidates = [Path.cwd() / 'data', Path.cwd().parent / 'data']
candidates += [p.parent for p in Path('/kaggle/input').glob('**/nomes.parquet')]
data = next(p for p in candidates if (p / 'nomes.parquet').exists())
names = pd.read_parquet(data / 'nomes.parquet')
surnames = pd.read_parquet(data / 'sobrenomes.parquet')
print(names.shape, surnames.shape)
names.head(3).T

## The most common names

Two different counts: people in the 2010 Census (IBGE) and company partners in the CNPJ register. Partners are adults who own companies, so the two rankings differ.

In [ ]:
top = names.nlargest(15, 'ibge_frequency').set_index('name')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
top['ibge_frequency'].div(1e6).plot.barh(ax=axes[0], color='#2a78d6', title='People, 2010 Census (millions)')
top['cnpj_partners'].div(1e3).plot.barh(ax=axes[1], color='#1f9e89', title='CNPJ partners (thousands)')
axes[0].invert_yaxis(); axes[0].set_ylabel('')
plt.tight_layout()

## Where each name is most common

`ibge_top_state` is the state where the name has the highest share of the population. Names typical of one state:

In [ ]:
(names.dropna(subset=['ibge_top_state'])
      .sort_values('ibge_top_state_per_100k', ascending=False)
      .groupby('ibge_top_state').head(1)
      .sort_values('ibge_top_state')[['ibge_top_state', 'name', 'ibge_top_state_per_100k', 'ibge_frequency']]
      .head(27))

## Surnames among company partners

In [ ]:
surnames.nlargest(20, 'cnpj_partners')[['surname', 'cnpj_partners']].set_index('surname').T

## Names that sound alike

`key_metaphone` is a Portuguese phonetic key: spellings with the same key sound alike.

In [ ]:
groups = (names.sort_values('ibge_frequency', ascending=False)
                .groupby('key_metaphone')
                .agg(spellings=('name', lambda s: ', '.join(s.head(8))), n=('name', 'size'),
                     people=('ibge_frequency', 'sum')))
groups[groups.n > 3].sort_values('people', ascending=False).head(10)

## Sex and decade of birth

From IBGE's v2 endpoint (2010 Census): how many women and men carry each spelling, and when they were born.

In [ ]:
if 'ibge_share_female' in names:
    common = names[names['ibge_frequency'] > 20000]
    print('Names used for both sexes (30-70% women):')
    display(common[common['ibge_share_female'].between(0.3, 0.7)]
            .sort_values('ibge_frequency', ascending=False)[['name', 'ibge_frequency', 'ibge_share_female']].head(10))
    decades = [c for c in names if c.startswith('ibge_born_')]
    trend = names.set_index('name').reindex(['Maria', 'José', 'Enzo', 'Valentina', 'Sebastião'])[decades].dropna(how='all')
    trend.columns = [c.replace('ibge_born_', '') for c in decades]
    trend.T.plot(figsize=(9, 4), marker='o', title='People alive in 2010, by decade of birth')
    plt.tight_layout()
else:
    print('This copy of the data has no sex and decade columns yet.')

## The texts: two models, two styles

Every `meaning` and `history` was written by a model and **never reviewed**. text-davinci-002 wrote short answers; gpt-3.5 writes longer ones and hedges more.

In [ ]:
texts = pd.concat([
    names.assign(table='names', length=names['meaning'].str.len())[['table', 'model', 'length']],
    surnames.assign(table='surnames', length=surnames['history'].str.len())[['table', 'model', 'length']],
])
texts.groupby(['table', 'model'])['length'].describe()[['count', '25%', '50%', '75%']].round(0)

In [ ]:
hedges = ['acredita-se', 'pode ser', 'possivelmente', 'provavelmente', 'é incerta']
for word in hedges:
    share = names.groupby('model')['meaning'].apply(lambda s: s.str.lower().str.contains(word).mean() * 100)
    print(f"{word:15s}", '  '.join(f"{m}: {v:4.1f}%" for m, v in share.items()))

## Wrong on purpose

These rows stay in the data as examples of hallucination.

In [ ]:
cases = {'Araken': 'Tupi origin, not Japanese', 'Shirley': 'not a variant of Cheryl'}
names[names['name'].isin(cases)][['name', 'model', 'meaning']]

In [ ]:
surnames[surnames['surname'].isin(['Cerqueira', 'Barboza', 'Carrero'])][['surname', 'model', 'history']]

## Templates the model repeats

The most frequent opening sentences show how formulaic the output is.

In [ ]:
import re

def template(row):
    first = re.split(r'(?<=[.!?])\s', row['meaning'], maxsplit=1)[0]
    return re.sub(r'\b' + re.escape(row['name']) + r'\b', 'X', first, flags=re.I)

names.apply(template, axis=1).value_counts().head(10)